# Dopamine PINN: mechanistic counterfactual analysis on the linear model

The causal-inference layer of the Level 2 research plan applied to the baseline linear
model, before the nonlinear extension. A reuptake inhibitor at fractional DAT occupancy
$A\in[0,1]$ acts as an intervention $do(A=a)$ on the uptake rate only:

$$\partial_t C = D\nabla^2 C - k(1-\eta A)\,C, \qquad D = 0.32,\; k = 0.020,\; \eta = 0.6 .$$

Training doses $A \in \{0, 0.25, 0.75, 1\}$ (400 scattered points each, 2% noise); $A = 0.5$
is never observed. Curriculum (`dopamine_pinn.causal.run_stages`): stage 2 $(D, k)$ from
untreated data, stage 3 $\eta$ with $(D, k)$ fixed, stage 4 joint, stage 5 held-out
$\tau_{0.5}$, $ATE(t)$, $CE(x,y)$, $GTE$, peak effect and $S_A = \partial C/\partial a$.
The reference estimator (FD least squares, Laplace) runs the same stages; its
delta-method SDs are the information limit every effect is compared with. Untreated
data with seed 4001 are exactly the benchmark set `linear_s4001`.

| Toggle | What | PINN studies | A100 time (est.) |
|---|---|---|---|
| `RUN_CANONICAL` | realization 4001, network seed 1234, figures | 1 | ~15 min |
| `RUN_CONFIRM` | realizations 4002, 4003 | 2 | ~30 min |
| `RUN_NET_SEEDS` | realization 4001, network seeds 1235-1238 (run-to-run spread) | 4 | ~1 h |

Results are saved after every study (to Google Drive with `USE_DRIVE = True`); a rerun
skips finished studies.

## 0. Install (Colab): pinned JAX stack + this repository as a package

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/Hachero98/dopamine-PINN-.git'
REPO_REF = 'causal-linear'   # branch or release tag
REPO_DIR = Path('/content/dopamine-PINN-')
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
SENTINEL = Path('/tmp/dopamine_pinn_ready_v1')

if IN_COLAB and not SENTINEL.exists():
    rc = os.system('pip -q install "jax[cuda12]==0.10.2" "flax==0.12.5" "optax>=0.2.3" "jaxopt>=0.8" scipy matplotlib')
    if not REPO_DIR.exists():
        rc |= os.system(f'git clone -q -b {REPO_REF} {REPO_URL} {REPO_DIR}')
    rc |= os.system(f'pip -q install --no-deps -e {REPO_DIR}')
    if rc != 0:
        raise RuntimeError('install failed - see the output above')
    SENTINEL.touch()
    print('Installed. Restarting the kernel; then Runtime -> Run all again.')
    os.kill(os.getpid(), 9)
else:
    print('Ready.')

In [ ]:
import json
import numpy as np
import jax
from IPython.display import Image, display

import dopamine_pinn as dp
from dopamine_pinn import config as cfg
from dopamine_pinn import causal as ca
from dopamine_pinn.pinn import TrainConfig
from dopamine_pinn.solver import make_fd_solver

print(f'dopamine_pinn {dp.__version__}  |  JAX {jax.__version__}  |  devices {jax.devices()}')

## 1. Toggles

In [ ]:
QUICK         = False   # tiny schedule for a smoke test (not results)
USE_DRIVE     = True
RUN_CANONICAL = True
RUN_CONFIRM   = True
RUN_NET_SEEDS = True

TC = TrainConfig.quick(verbose=True, log_every=4) if QUICK else TrainConfig()

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/dopamine_PINN_causal')
else:
    OUT = Path('causal_out')
if QUICK:
    OUT = OUT / 'quick'
OUT.mkdir(parents=True, exist_ok=True)
print(f'Results -> {OUT.resolve()}  |  Adam {TC.adam_iters}, L-BFGS {TC.lbfgs_iters} per stage')

SOLVER = make_fd_solver()


def public(d):
    if isinstance(d, dict):
        return {k: public(v) for k, v in d.items() if not str(k).startswith('_')}
    return d


def study(key, obs_seed, net_seed, figures=False):
    path = OUT / key / 'results.json'
    if path.exists():
        return json.loads(path.read_text())
    res, priv = ca.run_study(TC, obs_seed=obs_seed, seed=net_seed, pts_seed=cfg.SEED, solver=SOLVER)
    path.parent.mkdir(parents=True, exist_ok=True)
    if figures:
        ca.plot_study(res, priv, path.parent)
    path.write_text(json.dumps(public(res), indent=1, default=float))
    print(f'  {key}: {res["minutes"]:.1f} min')
    return json.loads(path.read_text())

## 2. Canonical realization 4001: identification, held-out counterfactual, figures

In [ ]:
if RUN_CANONICAL:
    can = study('obs4001_net1234', ca.OBS_SEED, cfg.SEED, figures=True)
    for name in ('parameter_recovery', 'effect_summaries', 'counterfactual_heatmaps'):
        display(Image(filename=str(OUT / 'obs4001_net1234' / f'{name}.png')))

## 3. Confirmation realizations and network-seed spread

In [ ]:
jobs = []
if RUN_CONFIRM:
    jobs += [(f'obs{s}_net1234', s, cfg.SEED) for s in (4002, 4003)]
if RUN_NET_SEEDS:
    jobs += [(f'obs4001_net{n}', ca.OBS_SEED, n) for n in (1235, 1236, 1237, 1238)]
for key, s, n in jobs:
    study(key, s, n)

## 4. Summary against the information limit

Signed errors (%) of the stage-4 parameters and stage-5 effects at $A = 0.5$. `ref ±SD`
is the reference estimator's Laplace SD (log scale for parameters, absolute GTE SD in
% of the true GTE): a PINN error inside about 2 SD is at the information limit of the
design.

In [ ]:
rows = []
for p in sorted(OUT.glob('obs*_net*/results.json')):
    r = json.loads(p.read_text())
    s5, gte_true = r['stage5'], r['stage5']['truth']['GTE']
    ref = s5.get('reference_calibrated_pde', {})
    rows.append({
        'study': p.parent.name,
        **{f'PINN {q}': r['pinn']['stage4']['err_pct'][q] for q in ca.PARAMS},
        **{f'ref {q}': r['reference']['stage4']['err_pct'][q] for q in ca.PARAMS},
        **{f'ref SD {q}': 100 * r['reference']['stage4']['sd_log'][q] for q in ca.PARAMS},
        'tau L2 net': s5['pinn_network']['E_L2_tau_pct'],
        'tau L2 PDE(PINN)': s5['pinn_calibrated_pde']['E_L2_tau_pct'],
        'tau L2 PDE(ref)': ref.get('E_L2_tau_pct', np.nan),
        'GTE net': s5['pinn_network']['GTE_err_pct'],
        'GTE PDE(PINN)': s5['pinn_calibrated_pde']['GTE_err_pct'],
        'GTE ref SD': 100 * ref.get('GTE_sd', np.nan) / gte_true,
        'S_A L2 net': s5['pinn_network']['E_L2_S_A_pct'],
    })
cols = list(rows[0]) if rows else []
print(' | '.join(f'{c:>14}' for c in cols))
for row in rows:
    print(' | '.join(f'{row[c]:>14}' if isinstance(row[c], str) else f'{row[c]:>14.2f}' for c in cols))
(OUT / 'summary.json').write_text(json.dumps(rows, indent=1, default=float))

In [ ]:
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive('/content/dopamine_PINN_causal', 'zip', OUT)
    files.download('/content/dopamine_PINN_causal.zip')